<a href="https://colab.research.google.com/github/adriatek/waymo-scene-verifier/blob/main/Waymo_Explorer.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install waymo-open-dataset-tf-2-12-0==1.6.7 --no-deps
!pip install protobuf==3.20.0

from waymo_open_dataset import dataset_pb2 as open_dataset
print("imports fine, we're in business")

In [ ]:
from google.colab import auth
auth.authenticate_user()

In [ ]:
gsutil ls gs://waymo_open_dataset_v_1_4_2/individual_files/training/ | head -5

In [ ]:


def read_in_segments():
  segment_files = ['gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10017090168044687777_6380_000_6400_000_with_camera_labels.tfrecord','gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10023947602400723454_1120_000_1140_000_with_camera_labels.tfrecord', 'gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-1005081002024129653_5313_150_5333_150_with_camera_labels.tfrecord','gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10061305430875486848_1080_000_1100_000_with_camera_labels.tfrecord','gs://waymo_open_dataset_v_1_4_2/individual_files/training/segment-10072140764565668044_4060_000_4080_000_with_camera_labels.tfrecord']

  for idx, file in enumerate(segment_files, start=1):
    !gsutil cp {file} /content/segment{idx}.tfrecord
    print(f"copied segment{idx}.tfrecord")



read_in_segments()

In [ ]:
from waymo_open_dataset import label_pb2

print(label_pb2.Label.Type.items())

In [ ]:
# extract first frame metadata into object and return it
import os
import pandas as pd
import tensorflow as tf
from waymo_open_dataset import dataset_pb2 as open_dataset

def extract_segment_metadata(filepath):
  segment_metadata = {}
  dataset = tf.data.TFRecordDataset(filepath, compression_type='')

  frame = open_dataset.Frame()
  for data in dataset:
    frame.ParseFromString(bytearray(data.numpy()))
    segment_metadata["segment_id"] = frame.context.name
    segment_metadata["time_of_day"] = frame.context.stats.time_of_day
    segment_metadata["weather"] = frame.context.stats.weather
    segment_metadata["location"] = frame.context.stats.location

    segment_metadata["vehicle_count"] = 0
    segment_metadata["pedestrian_count"] = 0
    segment_metadata["cyclist_count"] = 0

    for item in frame.context.stats.laser_object_counts:
      if item.type == 0: # TYPE UNKNOWN
        pass
      elif item.type == 1: # TYPE VEHICLE
        segment_metadata["vehicle_count"] += item.count
      elif item.type == 2: # TYPE PEDESTRIAN
        segment_metadata["pedestrian_count"] += item.count
      elif item.type == 3: # TYPE SIGN
        pass
      elif item.type == 4: # TYPE CYCLIST
        segment_metadata["cyclist_count"] += item.count
      else:
        pass

    return segment_metadata



folder = '/content'
file = 'segment3.tfrecord'
all_files = sorted(os.listdir(folder))

segment_file = os.path.join(folder,file)
extract_segment_metadata(segment_file)

rows = []
#for each segment, extract metadata object and store in rows array
for segment_file in all_files:
  if segment_file.endswith(".tfrecord"):
    segment_file = os.path.join(folder, segment_file)
    segment_metadata = extract_segment_metadata(segment_file)
    rows.append(segment_metadata)



In [ ]:
# Connect to the SQLite file in your Colab folder
import sqlite3

conn = sqlite3.connect('/content/waymo_segments.db')
segment_df = pd.DataFrame(rows)

segment_df.to_sql("segments", conn, if_exists="append", index=False)

# Now, write an SQL query on the table we made
query = "SELECT weather, COUNT(*) as total_segments FROM segments GROUP BY weather"
result = pd.read_sql_query(query, conn)

conn.close()

result.head()

# save as CSV
result.to_csv('/content/waymo_results.csv', index=False)
print("CSV file successfully saved in your Colab folder!")




In [ ]:
!pip install "protobuf==5.29.1"

In [ ]:
#what did the car see?
import matplotlib.pyplot as plt

img = frame.images[0]  # front camera
decoded = tf.image.decode_jpeg(img.image)

plt.figure(figsize=(15, 10))
plt.imshow(decoded)
plt.axis('off')
plt.show()

In [ ]:
import matplotlib.patches as patches

fig, ax = plt.subplots(figsize=(15,10))
ax.imshow(decoded)

#find the labels for this same camera
for camera_labels in frame.camera_labels:
  if camera_labels.name != img.name:
    continue
  for label in camera_labels.labels:
    rect = patches.Rectangle(
        (label.box.center_x - label.box.length / 2,
             label.box.center_y - label.box.width / 2),
            label.box.length, label.box.width,
            linewidth=2, edgecolor='red', facecolor='none')
    ax.add_patch(rect)

ax.axis('off')
plt.show()

